# Ejercicio 01 — Iranian Churn Dataset (UCI, id 563)

**Asignatura:** INF-8239 Ciencia de Datos II — Maestría en Ciencia de Datos e IA (UASD)
**Autora:** Michelle Cruz
**Unidad:** 01 — Modelos avanzados, reducción dimensional y Green AI

**Pregunta de análisis:** ¿Es posible predecir el abandono (`Churn`) de un cliente de
telecomunicaciones a partir de su comportamiento de uso durante los primeros meses de
servicio?

**Unidad de análisis:** un cliente/línea de un operador de telecomunicaciones.

**Target:** `Churn` (1 = abandonó, 0 = se mantuvo activo), medido a los 12 meses.

**Dataset:** Iranian Churn Dataset — UCI Machine Learning Repository, id 563,
licencia CC BY 4.0, DOI 10.24432/C5JW3Z. 3,150 filas × 13 variables predictoras + target.

**Advertencia:** este ejercicio es un ejemplo académico de clasificación; no constituye
una recomendación operativa de retención sin validación adicional del negocio.


## 0. Ubicar el proyecto en Google Drive (una sola vez)

**La primera vez** que corras esta celda: te pedirá conectar tu cuenta de Google (aparece
un enlace de autorización) y luego subir `INF8239_U01_actualizado.zip` — lo descomprime
directamente dentro de tu Drive, en `Mi unidad/INF8239_U01`.

**Las siguientes veces** (nueva sesión de Colab, otro día): solo monta tu Drive y encuentra
el proyecto automáticamente — **no vuelve a pedir el zip**, porque ya quedó guardado ahí de
forma permanente.

**Si trabajas en Jupyter local** (no Colab): no pasa nada distinto, sigue detectando la
carpeta del proyecto automáticamente sin necesidad de Drive.

In [ ]:
import sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
MARKER = "src/inf8239_u01"


def find_project_root(marker=MARKER, start=None):
    here = start or Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive")

    PROJECT_ROOT = find_project_root(start=DRIVE_BASE)

    if PROJECT_ROOT is None:
        # Primera vez: sube el zip y lo descomprime DENTRO de Drive (queda permanente)
        from google.colab import files

        print("No se encontró el proyecto en tu Drive todavía.")
        print("Selecciona INF8239_U01_actualizado.zip para guardarlo en tu Drive:")
        subido = files.upload()
        nombre_zip = next(iter(subido))

        with zipfile.ZipFile(nombre_zip, "r") as zf:
            zf.extractall(DRIVE_BASE)

        PROJECT_ROOT = find_project_root(start=DRIVE_BASE)
        if PROJECT_ROOT is None:
            raise FileNotFoundError(
                "El zip se descomprimió en tu Drive pero no se encontró "
                "'src/inf8239_u01' dentro. Verifica que subiste el archivo correcto."
            )
        print("Proyecto guardado permanentemente en:", PROJECT_ROOT)
    else:
        print("Proyecto encontrado en tu Drive:", PROJECT_ROOT)
else:
    PROJECT_ROOT = find_project_root()
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"No se encontro la carpeta '{MARKER}' subiendo desde {Path.cwd()}.\n"
            "Verifica que este notebook este dentro de la carpeta del proyecto "
            "INF8239_U01 (en cualquier subcarpeta), y que 'src/inf8239_u01' exista."
        )

sys.path.append(str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT:", PROJECT_ROOT)


## 1. Carga del Iranian Churn Dataset

### 1.1 Subir el CSV del dataset (solo la primera vez)

El dataset **no viene incluido en el zip** (debes descargarlo tú de UCI). Si ya lo subiste
antes, esta celda lo detecta en tu Drive y no vuelve a pedirlo. Si es la primera vez, te
pedirá el archivo y lo guardará dentro del proyecto en tu Drive, de forma permanente.

In [ ]:
dataset_destino = PROJECT_ROOT / "data" / "raw" / "iranian_churn_dataset.csv"

if IN_COLAB and not dataset_destino.exists():
    from google.colab import files

    print("No se encontró el CSV del dataset. Selecciona tu archivo descargado de UCI:")
    subido_csv = files.upload()
    nombre_csv = next(iter(subido_csv))

    dataset_destino.parent.mkdir(parents=True, exist_ok=True)
    Path(nombre_csv).rename(dataset_destino)
    print("Dataset guardado en:", dataset_destino)
else:
    print("Dataset ya presente en:", dataset_destino)


In [ ]:
import pandas as pd
from inf8239_u01.data import download_csv

iranian_churn_df = pd.read_csv(dataset_destino)

print(iranian_churn_df.shape)
iranian_churn_df.head()


## 2. Auditoría antes de modelar

In [ ]:
iranian_churn_audit = pd.DataFrame({
    "tipo": iranian_churn_df.dtypes.astype(str),
    "ausentes": iranian_churn_df.isna().sum(),
    "porcentaje_ausente": (iranian_churn_df.isna().mean() * 100).round(2),
    "unicos": iranian_churn_df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("Duplicados:", iranian_churn_df.duplicated().sum())
print("Porcentaje duplicados:", round(iranian_churn_df.duplicated().mean() * 100, 2), "%")
iranian_churn_audit


**Decisión sobre duplicados:** la literatura sobre este dataset reporta ~9.5 % de
registros duplicados exactos. No se eliminan automáticamente: documenta aquí tu decisión
final (conservarlos o eliminarlos) y la justificación semántica, antes de continuar.

## 3. Definición del target y variables

In [ ]:
TARGET = "Churn"
DROP_COLUMNS = []  # No hay columnas de ID en este dataset; Status se conserva
                    # y se analiza aparte en la Sección 8 (fuga de datos)

assert TARGET in iranian_churn_df.columns

X_iranian_churn = iranian_churn_df.drop(columns=[TARGET] + DROP_COLUMNS)
y_iranian_churn = iranian_churn_df[TARGET]

print(y_iranian_churn.value_counts(normalize=True).round(3))
assert y_iranian_churn.notna().all()
assert y_iranian_churn.nunique() >= 2


## 4. Partición train/test (estratificada, fija para todo el ejercicio)

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X_iranian_churn, y_iranian_churn,
    test_size=0.20, random_state=42, stratify=y_iranian_churn
)

print(Xtr.shape, Xte.shape)
print(ytr.value_counts(normalize=True).round(3))
print(yte.value_counts(normalize=True).round(3))


## 5. Preprocesamiento (dentro de Pipeline, sin fuga)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = Xtr.select_dtypes(include="number").columns.tolist()
cat_cols = Xtr.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocess_iranian_churn = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print("Numéricas:", len(num_cols), "| Categóricas:", len(cat_cols))


## 6. Línea base (Dummy) vs. SVM

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

dummy_iranian_churn = Pipeline([
    ("prep", preprocess_iranian_churn),
    ("model", DummyClassifier(strategy="most_frequent"))
])
svm_iranian_churn = Pipeline([
    ("prep", preprocess_iranian_churn),
    ("model", SVC(C=1, gamma="scale", probability=True, random_state=42))
])

for nombre, modelo in {"dummy": dummy_iranian_churn, "svm": svm_iranian_churn}.items():
    modelo.fit(Xtr, ytr)
    pred = modelo.predict(Xte)
    print(nombre, "F1 macro:", round(f1_score(yte, pred, average="macro"), 4))

print()
print(classification_report(yte, svm_iranian_churn.predict(Xte), digits=4))


## 7. Búsqueda de hiperparámetros (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_iranian_churn = GridSearchCV(
    estimator=svm_iranian_churn,
    param_grid={
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1]
    },
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True
)
search_iranian_churn.fit(Xtr, ytr)

print(search_iranian_churn.best_params_)
print("CV F1 macro:", round(search_iranian_churn.best_score_, 4))

mejor_svm_iranian_churn = search_iranian_churn.best_estimator_
pred_mejor = mejor_svm_iranian_churn.predict(Xte)
print(classification_report(yte, pred_mejor, digits=4))


## 8. Análisis de fuga de datos — variable `Status`

**Contexto:** `Status` indica si la cuenta estaba activa a los **9 meses**, mientras que
`Churn` (el target) mide el abandono a los **12 meses**. No es una fuga idéntica, pero está
fuertemente correlacionada con el target porque mide el mismo fenómeno en un momento
anterior. Se conserva la variable, pero se compara el modelo **con** y **sin** ella para
documentar cuánto depende el resultado de esa señal cercana a la respuesta.

In [ ]:
# --- Comparación de fuga de datos: Iranian Churn Dataset ---
# Variable en revisión: "Status" (activo/inactivo a los 9 meses) vs. Churn (12 meses)

# 1. Versión CON "Status" (13 variables, dataset completo) -> ya calculada arriba
pred_con_status = mejor_svm_iranian_churn.predict(Xte)

# 2. Versión SIN "Status" (12 variables)
Xtr_sin_status = Xtr.drop(columns=["Status"])
Xte_sin_status = Xte.drop(columns=["Status"])

num_cols_sin_status = Xtr_sin_status.select_dtypes(include="number").columns.tolist()
cat_cols_sin_status = Xtr_sin_status.select_dtypes(exclude="number").columns.tolist()

preprocess_iranian_churn_sin_status = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), num_cols_sin_status),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols_sin_status)
])

svm_iranian_churn_sin_status = Pipeline([
    ("prep", preprocess_iranian_churn_sin_status),
    ("model", SVC(C=search_iranian_churn.best_params_["model__C"],
                  gamma=search_iranian_churn.best_params_["model__gamma"],
                  probability=True, random_state=42))
])
svm_iranian_churn_sin_status.fit(Xtr_sin_status, ytr)
pred_sin_status = svm_iranian_churn_sin_status.predict(Xte_sin_status)

# 3. Comparar resultados
from sklearn.metrics import recall_score

comparacion_fuga_iranian_churn = pd.DataFrame({
    "version": ["Con Status", "Sin Status"],
    "f1_macro": [
        f1_score(yte, pred_con_status, average="macro"),
        f1_score(yte, pred_sin_status, average="macro")
    ],
    "recall_macro": [
        recall_score(yte, pred_con_status, average="macro"),
        recall_score(yte, pred_sin_status, average="macro")
    ]
})

print("=== Comparación de fuga de datos - Iranian Churn Dataset ===")
print(comparacion_fuga_iranian_churn)

print()
print("--- Reporte CON Status ---")
print(classification_report(yte, pred_con_status, digits=4))

print()
print("--- Reporte SIN Status ---")
print(classification_report(yte, pred_sin_status, digits=4))


In [ ]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(exist_ok=True)

comparacion_fuga_iranian_churn.to_csv(
    reports_dir / "iranian_churn_leakage_status_comparison.csv", index=False
)
comparacion_fuga_iranian_churn


**Interpretación (completar tras ejecutar):**

- Diferencia de F1-macro entre ambas versiones: _____
- ¿La caída al quitar `Status` es grande o pequeña? _____
- Decisión final (¿se conserva `Status` o se retira?) y justificación de negocio: _____


## 9. Guardar resultados finales y versionar

In [ ]:
import joblib

joblib.dump(mejor_svm_iranian_churn, reports_dir / "iranian_churn_svm_best.joblib")

# Desde una terminal, en la raíz del proyecto:
# git add .
# git commit -m "feat: comparar fuga de datos Status en Iranian Churn Dataset"
print("Modelo guardado en:", reports_dir / "iranian_churn_svm_best.joblib")


## 10. Conclusión (300 a 500 palabras)

_Completar aquí: dataset usado, decisión sobre duplicados, resultado de la búsqueda de
hiperparámetros, resultado de la comparación de fuga de datos (`Status`), decisión final
sobre las variables, limitaciones y contexto de aplicación (retención de clientes en un
operador de telecomunicaciones)._
